# Questions & Answers
in this notebook we'll answer 10 questions from our database
and answer them with queries

In [1]:
#setup
import atexit
import sqlite3
from pathlib import Path
import pandas as pd

In [2]:
DB = Path("music.db").resolve() # Database file path

In [3]:
# Connect to database 
conn = sqlite3.connect(DB)
conn.execute("PRAGMA foreign_keys = ON;")
conn.execute("PRAGMA journal_mode = WAL;")
conn.execute("PRAGMA busy_timeout = 5000;")
_ = atexit.register(conn.close)

In [4]:
# This function will run sql queries and return the answer as a pandas DataFrame
def query_df(sql: str, params=()) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params)

In [5]:
pd.set_option('display.max_rows', None)      # show all rows
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.max_colwidth', None)  # don't cut long text cells
pd.set_option('display.width', None)         # don't wrap wide tables

# Question 1
Do explicit tracks cluster in high-danceability genres, or is explicitness itself linked to danceability?

In [6]:
answer_q1 = query_df("""
    SELECT m.explicit,
        COUNT(*) AS n_tracks,
        ROUND(AVG(a.danceability), 4) AS avg_danceability
    FROM tracks_metadata AS m
    JOIN audio_features AS a
    ON a.track_key = m.track_key
    GROUP BY m.explicit
    ORDER BY m.explicit
""")
answer_q1

,explicit,n_tracks,avg_danceability
0,0,76349,0.5541
1,1,7107,0.6235


# Answer question 1
Explicit tracks average 0.624 danceability vs. 0.554 for clean — a gap of ~0.07, about 0.4 standard deviations. This aggregate does not distinguish between explicitness itself and the genres explicit tracks cluster in; a per-genre breakdown would test that.

# Question 2

In [7]:
answer_q2 = query_df("""
    SELECT g.track_genre,
        COUNT(*) AS n_tracks,
        ROUND(AVG(a.energy), 4) AS avg_energy,
        ROUND(SQRT(
            AVG(a.energy * a.energy)
            - AVG(a.energy) * AVG(a.energy)
        ), 4) AS sd_energy
    FROM track_genres AS g
    JOIN audio_features a
    ON a.track_key = g.track_key
    GROUP BY g.track_genre
    HAVING COUNT(*) >= 100
    ORDER BY sd_energy DESC
""")
answer_q2

,track_genre,n_tracks,avg_energy,sd_energy
0,sleep,912,0.3466,0.3819
1,iranian,973,0.5446,0.3361
2,piano,930,0.2999,0.2978
3,german,802,0.5255,0.2966
4,anime,987,0.6721,0.2756
5,idm,982,0.5543,0.2593
6,british,948,0.5056,0.2469
7,world-music,983,0.5330,0.2446
8,children,936,0.4964,0.2399
9,acoustic,958,0.4365,0.2366


# Answer question 2
The widest internal energy spread belongs not to musical styles but to label categories: sleep (SD 0.38), iranian, piano, german. The narrowest are tight subgenres: hardstyle (SD 0.08), death-metal, metalcore. track_genre is not a taxonomy — it mixes musical styles with countries, moods, instruments, and media franchises. Comparing "hardstyle" to "iranian" is not comparing like with like.

# Question 3

In [8]:
answer_q3 = query_df("""
    SELECT g.track_genre,
        AVG(a.loudness) AS avg_loudness,
        AVG(a.energy) AS avg_energy,
        AVG(a.energy) / ABS(AVG(a.loudness)) AS energy_per_db
    FROM track_genres AS g
    JOIN audio_features AS a
    ON g.track_key = a.track_key
    GROUP BY g.track_genre
    ORDER BY energy_per_db DESC
""")
answer_q3

,track_genre,avg_loudness,avg_energy,energy_per_db
0,j-idol,-3.788523,0.868256,0.229181
1,hardstyle,-4.344132,0.901444,0.207509
2,drum-and-bass,-4.289020,0.876697,0.204405
3,metalcore,-4.472823,0.913755,0.204291
4,party,-4.459625,0.867897,0.194612
5,death-metal,-5.341623,0.931504,0.174386
6,happy,-5.444310,0.910553,0.167249
7,forro,-5.134633,0.789259,0.153713
8,heavy-metal,-5.727701,0.874715,0.152717
9,metal,-5.508455,0.835732,0.151718


# Answer question 3
Loudness and energy are coupled: j-idol (0.229), hardstyle (0.208), drum-and-bass (0.204), metalcore (0.204), and death-metal (0.174) rank highest on energy-per-decibel — loud and energetic. The dynamic end is the mirror image: ambient (0.013), new-age (0.011), and classical (0.010) sit ~20× lower on the same ratio. This is only a rough proxy — a true compression measure needs dynamic-range data Spotify does not expose.

# Question 4

In [9]:
answer_q4 = query_df("""
    SELECT
        CASE
            WHEN duration_m < 2.5 THEN '1_under_2.5m'
            WHEN duration_m < 3.5 THEN '2_2.5_to_3.5m'
            WHEN duration_m < 4.5 THEN '3_3.5_to_4.5m'
            ELSE 'over_4.5m'
        END AS duration_bucket,
        COUNT(*) AS n_tracks,
        AVG(popularity) AS avg_popularity
    FROM tracks_metadata AS m
    GROUP BY duration_bucket
    ORDER BY duration_bucket
""")
answer_q4

,duration_bucket,n_tracks,avg_popularity
0,1_under_2.5m,10888,31.714732
1,2_2.5_to_3.5m,26406,36.660910
2,3_3.5_to_4.5m,25357,37.034271
3,over_4.5m,20805,32.223696


# Answer question 4
Popularity peaks in the 2.5–4.5 minute range (36.7–37.0) and drops by ~5 points at both extremes. The "shorter is better" narrative does not hold here: under-2.5-minute tracks are the least popular bucket (31.7), because very short tracks in this dataset are interludes and snippets, not songs. The relationship is an inverted U, not a monotonic preference for brevity.

# Question 5

In [10]:
answer_q5 = query_df("""
    SELECT
        CASE
            WHEN a.valence < 0.2 THEN '1_very_sad'
            WHEN a.valence < 0.4 THEN '2_sad'
            WHEN a.valence < 0.6 THEN '3_neutral'
            WHEN a.valence < 0.8 THEN '4_happy'
            ELSE '5_very_happy'
        END AS valence_bucket,
        COUNT(*) AS n_tracks,
        AVG(m.popularity) AS avg_popularity
    FROM audio_features AS a
    JOIN tracks_metadata AS m
    ON a.track_key = m.track_key
    GROUP BY valence_bucket
    ORDER BY valence_bucket
""")
answer_q5

,valence_bucket,n_tracks,avg_popularity
0,1_very_sad,16356,32.601736
1,2_sad,20404,36.329200
2,3_neutral,19027,36.338256
3,4_happy,16352,35.238931
4,5_very_happy,11317,33.643192


# Answer question 5
Popularity is highest for moderate valence — sad and neutral tie at ~36.3 — and drops at both extremes: very_sad is the least popular bucket (32.6), very_happy second-least (33.6). This directly contradicts the "sad songs win" narrative. The data shows a preference for emotional middle ground, not for misery.

# Question 6

In [11]:
answer_q6 = query_df("""
    SELECT 
        g.track_genre,
        AVG(a.instrumentalness) AS avg_instrumentalness,
        AVG(m.popularity) AS avg_popularity,
        COUNT(*) AS n_tracks
    FROM track_genres AS g
    JOIN audio_features AS a
    ON a.track_key = g.track_key
    JOIN tracks_metadata AS m
    ON g.track_key = m.track_key
    GROUP BY g.track_genre
    HAVING AVG(a.instrumentalness) > 0.5
    ORDER BY avg_instrumentalness DESC
""")
answer_q6

,track_genre,avg_instrumentalness,avg_popularity,n_tracks
0,study,0.789227,26.152610,996
1,minimal-techno,0.746567,34.435950,968
2,sleep,0.731758,36.134868,912
3,new-age,0.716865,29.296146,986
4,detroit-techno,0.700207,11.199393,988
5,idm,0.680257,15.819756,982
6,ambient,0.676669,45.002079,962
7,classical,0.594608,15.183047,814
8,iranian,0.590734,2.250771,973
9,techno,0.549378,39.766940,974


# Answer question 6
Instrumental-heavy genres average ~8 points below the dataset mean (~27 vs. 35), but the spread is enormous — from piano at 48 and ambient at 45 down to iranian at 2.3 and detroit-techno at 11.2. Instrumentalness alone does not predict popularity; it is confounded with genre and use-case. Instrumental piano succeeds because it serves a purpose (study, focus); instrumental classical stays niche because it demands attention.

# Question 7

In [12]:
answer_q7 = query_df("""
    SELECT
        CASE
            WHEN a.tempo < 60   THEN '1_under_60'
            WHEN a.tempo < 90   THEN '2_60_to_90'
            WHEN a.tempo < 120  THEN '3_90_to_120'
            WHEN a.tempo < 150  THEN '4_120_to_150'
            WHEN a.tempo < 180  THEN '5_150_to_180'
            ELSE                     '6_over_180'
        END AS tempo_bucket,
        COUNT(*)                AS n_tracks,
        AVG(a.danceability)     AS avg_danceability
    FROM audio_features a
    GROUP BY tempo_bucket
    ORDER BY tempo_bucket;
""")
answer_q7

,tempo_bucket,n_tracks,avg_danceability
0,1_under_60,307,0.346906
1,2_60_to_90,11905,0.480215
2,3_90_to_120,26543,0.597898
3,4_120_to_150,30005,0.593285
4,5_150_to_180,12290,0.502068
5,6_over_180,2406,0.444022


# Answer question 7
Danceability peaks at 90–150 BPM (0.598 at 90–120, 0.593 at 120–150 — effectively tied), then falls off sharply: −0.12 below 90 BPM and −0.10 to −0.15 above 150 BPM. This is a genuine inverted U, not a linear trend — the "faster = more danceable" shorthand overstates the case. The tiny sub-60 bucket (307 tracks) is ambient/sleep material, not slow songs, and should not be read as evidence about tempo.

# Question 8

In [13]:
# Question 8 part a — Global means (single row)
answer_q8_global = query_df("""
    SELECT
        AVG(a.acousticness) AS avg_acousticness,
        AVG(a.energy)       AS avg_energy
    FROM audio_features a
""")
answer_q8_global

,avg_acousticness,avg_energy
0,0.328816,0.635771


In [14]:
# Question 8 part b — Per-genre view (one row per genre)
answer_q8_genre = query_df("""
    SELECT g.track_genre,
           AVG(a.acousticness)  AS avg_acousticness,
           AVG(a.energy)        AS avg_energy,
           COUNT(*)             AS n
    FROM track_genres g
    JOIN audio_features a ON a.track_key = g.track_key
    GROUP BY g.track_genre
    ORDER BY avg_acousticness DESC
""")
answer_q8_genre

,track_genre,avg_acousticness,avg_energy,n
0,classical,0.912601,0.201283,814
1,romance,0.866422,0.298650,887
2,tango,0.846467,0.372079,996
3,new-age,0.825138,0.212917,986
4,opera,0.797384,0.313572,961
5,guitar,0.774048,0.322843,985
6,ambient,0.773472,0.239060,962
7,disney,0.746032,0.305650,923
8,comedy,0.726427,0.699893,996
9,piano,0.724901,0.299916,930


# Answer question 8
Acousticness and energy form the cleanest structural gradient in the dataset: classical at 0.91 / 0.20 energy down to death-metal at 0.007 / 0.93, in near-perfect monotonic order. The correlation is strong at the genre level, but not universal — comedy (0.73 / 0.70), pagode (0.55 / 0.71), and salsa (0.47 / 0.72) break it. Comedy is a category error (spoken word); pagode and salsa are genuine counterexamples where acoustic instruments and high rhythmic energy coexist because energy comes from arrangement, not timbre.

# Question 9

In [15]:
# Q9 — Major vs minor valence (the fixed query)
answer_q9 = query_df("""
    SELECT a.mode,
           COUNT(*)                    AS n,
           ROUND(AVG(a.valence), 4)    AS avg_valence
    FROM audio_features a
    GROUP BY a.mode
    ORDER BY a.mode
""")
answer_q9

,mode,n,avg_valence
0,0,30561,0.4570
1,1,52895,0.4705


# Answer question 9
Major-key tracks average 0.4705 valence against 0.4570 for minor-key — a gap of 0.0135, about 5% of a standard deviation. This is statistically detectable at 83k rows but practically meaningless: mode tells you almost nothing about how positive a produced track sounds. The classic "major = happy, minor = sad" result holds for isolated chords, not full productions, where arrangement and tempo swamp the modal signal.

# Question 10

In [16]:
answer_q10 = query_df("""
    SELECT
        g.track_genre,
        AVG(a.energy)     AS avg_energy,
        AVG(a.valence)    AS avg_valence,
        AVG(m.popularity) AS avg_popularity,
        COUNT(*)          AS n
    FROM track_genres g
    JOIN audio_features a ON a.track_key = g.track_key
    JOIN tracks_metadata m ON m.track_key = a.track_key
    GROUP BY g.track_genre
    HAVING AVG(a.energy) > 0.65
        AND AVG(a.valence) < 0.35
    ORDER BY avg_popularity DESC;
""")
answer_q10

,track_genre,avg_energy,avg_valence,avg_popularity,n
0,metalcore,0.913755,0.316623,43.713693,964
1,dubstep,0.759893,0.271428,43.068277,952
2,techno,0.744454,0.318866,39.766940,974
3,trance,0.847766,0.265678,39.262313,934
4,minimal-techno,0.677848,0.282446,34.435950,968
5,death-metal,0.931504,0.248220,32.085945,989
6,drum-and-bass,0.876697,0.321126,26.721480,973
7,hardstyle,0.901444,0.313919,26.673136,979
8,black-metal,0.874455,0.191868,22.411883,993
9,happy,0.910553,0.328996,21.845915,967


# Answer question 10
Sad-energy genres (energy > 0.65, valence < 0.35) average ~31.3 popularity vs. a dataset mean of 35.0 — as a group they are less popular, not more. But the spread is the real finding: metalcore (43.7) and dubstep (43.1) sit well above the mean, while grindcore (14.6) and black-metal (22.4) sit far below. The quadrant is not one market — it is several subcultures with opposite commercial trajectories.